In [ ]:
import tomllib
from pathlib import Path

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import arya
arya.init("mnras")

In [ ]:
from sparcl.client import SparclClient

In [ ]:
from astropy.table import Table

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils
from desi_utils.plotting import plot_wrapped_spectrum
from desi_utils import EMISSION_LINES, get_wavelength

In [ ]:
from scipy.ndimage import gaussian_filter1d

In [ ]:
CA_TRIPLET = np.array([8500.36, 8544.44, 8664.52] )

In [ ]:
MG_TRIPLET = np.array([5168.76, 5174.12, 5185.05])

In [ ]:
CA_II_K = 3934.78

In [ ]:
NA_DOUBLET = np.array([5891.58, 5897.56])

# Load spectrum

In [ ]:
with open("../analysis/selected_targetid.txt", "rb") as f:
    targetids = tomllib.load(f)["targetid"]

In [ ]:
idx = 1

In [ ]:
targetid = targetids[idx]

In [ ]:
galspec_raw = desi_utils.get_data.get_spectra(targetid)

In [ ]:
if len(galspec_raw.flux.shape) == 2:
    galspec_raw= galspec_raw[0]
    for key in galspec_raw.meta.keys():
        if key in ["original_wcs"]:
            continue
        galspec_raw.meta[key] = galspec_raw.meta[key][0]

    

In [ ]:
spec = desi_utils.Spectrum.from_desi(galspec_raw)

In [ ]:
out_dir = Path(".", "example_fits", str(targetid))

In [ ]:
if not out_dir.exists():
    out_dir.mkdir()

In [ ]:
skyspec_table = Table.read("../data/skyspectrum.fits")
skyspec = desi_utils.Spectrum(wavelength=skyspec_table["wavelength"], flux=skyspec_table["flux"], uncertainty=0)

# Plot spectrum

In [ ]:
def plot_lines(spec, lines, restframe=True):

    if restframe:
        specwave = spec.wavelength / (1 + spec.redshift)
    else:
        specwave = spec.wavelength
        
    for line in lines:
        wave = desi_utils.get_wavelength(line)
        if not restframe:
            wave *= 1 + spec.redshift
            
        idx = np.argmin(np.abs(specwave - wave))
        y0 = np.max(spec.flux[max(0, idx-5):min(idx+5, len(specwave))])

        plt.annotate(line.replace("_", ""), (wave, y0), (0, 2), 
                     xycoords="data", textcoords="offset fontsize", 
                     fontsize=6, rotation=90, ha="center",
                     arrowprops=dict(arrowstyle="-", linewidth=1, color="black"),
                    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2))

plot_lines(spec, desi_utils.EMISSION_LINES, restframe=False)
spec.plot(plot_uncertainty=True)

plt.savefig(out_dir / "obs_spec_compressed.png")

In [ ]:
specdict = {
    "observations": spec,
    # "smoothed spectrum": (spec_resamp.wavelength, convolve(spec_resamp.flux, smoothing_kernel)),
    "sky": (skyspec.wavelength, 0.1*skyspec.flux) ,
    "std": (spec.wavelength, spec.uncertainty),
}
plot_wrapped_spectrum(
    {"observations": spec},
    redshift=spec.redshift, label_lines=desi_utils.EMISSION_LINES, show_uncertainty=True,
)

plt.savefig(out_dir / "obs_spec.png")

In [ ]:
def plot_stellar_line(spec, lines, label):
    fig, ax = plt.subplots(figsize=(8, 2))

    xlim = np.min(lines)-50, np.max(lines) + 50
    spec.plot(xlim=xlim, plot_uncertainty=True, restframe=True, label="observed spectrum")
  
    y0 = np.min(spec.flux[(spec.wavelength > xlim[0] ) & (xlim[1] > spec.wavelength)])
    y0 -= 5
    plt.scatter(lines, np.full_like(lines, y0), color="C2", s=10)
    plt.annotate(label, (np.max(lines), y0), (1, 0), color="C2", xycoords="data", textcoords="offset fontsize", va="center")



In [ ]:
np.median(spec.flux / spec.uncertainty)

In [ ]:
plot_stellar_line(spec, MG_TRIPLET, "Mg triplet")

In [ ]:
plot_stellar_line(spec, NA_DOUBLET, "Na I D")

In [ ]:
plot_stellar_line(spec, CA_TRIPLET, "Ca triplet")

In [ ]:
plot_stellar_line(spec, [CA_II_K], "Ca II K")